# tokens

> The styling surface: colour, space, radius, type, motion, and elevation

In [ ]:
#| default_exp tokens

Tokens are the shared values that components use for styling. Changing a token changes all the components that read it. We register each family with `reg.rule` in the `tokens` layer, leaving applications free to override it with ordinary CSS.

In [ ]:
from importlib.resources import files
from fastcore.utils import *
from answerui.registry import reg, Registry

In [ ]:
from fastcore.test import *
from fasthtml.common import Div
from answerui.demo import demo


## Colour

The palette notebook produced twelve CSS custom properties: neutral and vivid ladders derived from `--hue`, with light and dark values for each. Here we give them an application default and named colour contexts.

In [ ]:
PALETTE = files('answerui').joinpath('css/palette.css').read_text()

In [ ]:
print(f'{(_ := PALETTE.splitlines())[0][:100]}…\n…\n{_[-1][:100]}…')

--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.0032…
…
--v2: light-dark(oklch(0.4111 clamp(0, 0.12402 + 0.05381*cos((1*var(--hue) - 328.74)*1deg) + 0.02534…


In [ ]:
rungs = re.findall(r'^--([\w-]+):', PALETTE, re.M)
test_eq(len(rungs), 12)


### Application palette

We use hue 260 as the default. `color-scheme: light dark` lets the browser follow the user's preference.

In [ ]:
_hue_css = ':root { --hue: 260; color-scheme: light dark; }'

### Colour contexts

Danger, warning, success, and information each get a hue and a bare attribute. These hues are independent of the application hue.

In [ ]:
CONTEXTS = dict(danger=25, warning=75, success=145, info=230)

A custom property is substituted where it is declared, so an element that sets `--hue` inherits rungs already derived from its parent's hue. A colour context therefore needs two things: `--hue` re-pointed on its selector, and the ladder declared on that same selector. A registry's `context` records both, and the palette rule is built from every context declared whenever the stylesheet is assembled, so a later module or an application can add a context without repeating the ladder:


In [ ]:
@patch(as_prop=True)
def contexts(self:Registry):
    "The colour contexts declared on this registry: selector to hue"
    return self.__dict__.setdefault('_contexts', {})

@patch
def context(
    self:Registry,
    sel, # Selector of the context
    hue, # Its hue: a number, or an expression such as `var(--hue-danger)`
):
    "Declare `sel` as a colour context whose family derives from `hue`"
    self.contexts[sel] = hue


In [ ]:
def _context_css(r): return '\n'.join(f'{sel} {{ --hue: {hue}; }}' for sel,hue in r.contexts.items())
def _palette_css(r): return f"{_context_css(r)}\n{', '.join([':root', *r.contexts])} {{\n{PALETTE}\n}}"


The four severity contexts are the first:

In [ ]:
for name in CONTEXTS: reg.context(f'[{name}]', f'var(--hue-{name})')

In [ ]:
print(_context_css(reg))

[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }


We register the application default, context hues, and palette declarations in the `tokens` layer.

In [ ]:
_hues = ' '.join(f'--hue-{name}: {hue};' for name,hue in CONTEXTS.items())
reg.rule('hue', _hue_css, layer='tokens', doc='The application hue and the supported colour schemes.')
reg.rule('contexts', f':root {{ {_hues} }}', layer='tokens', doc='The severity hues, each independently adjustable.')
reg.rule('palette', partial(_palette_css, reg), layer='tokens', doc='The neutral and vivid ladders, declared on the root and on every colour context.')


palette [tokens]: The neutral and vivid ladders, declared on the root and on every colour context.

To see the tokens, a helper binds a list of token names to one CSS property through inline styles:

In [ ]:
def swatches(prop, names, **kw):
    "Tiles binding each named token to `prop`"
    tiles = [Div(n, style=f'padding:.5rem;background:var(--n-2);font:12px monospace;{prop}:var(--{n})') for n in names]
    return Div(*tiles, style='display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)', **kw)

The ladders under the application hue, in a danger context, and as a dark island:

In [ ]:
demo(swatches('background', rungs), swatches('background', rungs, danger=True),
    Div(swatches('background', rungs), style='color-scheme:dark'))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }
:root, [danger], [warning], [success], [info], [project], mark {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 + 0.03348*cos((1*var(--hue) - 339.18)*1deg) + 0.01411*cos((2*var

## Typography

Fonts and their sizes are defined here. Five sizes derive from the body size, `--text`. The base is declared on the root and the sizes on every element, so a scope that sets `--text` gets a scale of its own, and everything inside it inherits the result. Text inherits its size rather than reading the token, so such a scope also sets `font-size` from `--text`. We also declare regular and semibold weights, plus normal and tight line heights.

These are shared values, not element styles: the base layer will style native elements, while typrose will handle prose regions.

In [ ]:
_type_css = r"""
:root {
  --font: system-ui, sans-serif;
  --font-mono: ui-monospace, Menlo, Consolas, monospace;
  --text: 1rem;
  --weight: 400; --weight1: 600;
  --leading-1: 1.25; --leading: 1.5;
}
*, ::before, ::after {
  --text-1: calc(var(--text) * .875);
  --text1: calc(var(--text) * 1.25);
  --text2: calc(var(--text) * 1.5);
  --text3: calc(var(--text) * 2);
}
"""

In [ ]:
reg.rule('type', _type_css, layer='tokens',
    doc='System font families, five sizes, regular and semibold weights, and two line heights.')

type [tokens]: System font families, five sizes, regular and semibold weights, and two line heights.

In [ ]:
demo(swatches('font-size', ['text-1', 'text', 'text1', 'text2', 'text3']))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }
:root, [danger], [warning], [success], [info], [project], mark {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 + 0.03348*cos((1*var(--hue) - 339.18)*1deg) + 0.01411*cos((2*var

## Space

One scale spaces component interiors and the gaps between components. Each step derives from `--space` on every element, so changing the base on `:root` adjusts spacing across the application, and changing it on one region gives that region a scale of its own. Components use the scale for internal padding; layout parents use it for gaps, rather than components giving themselves outer margins.

The base value is the control and the steps are what components read. Setting a step directly on a region changes that element alone, since its descendants derive their own from the base they inherit.

In [ ]:
reg.rule('space', r"""
:root { --space: 1rem; }
*, ::before, ::after {
  --space-3: calc(var(--space) * .25);
  --space-2: calc(var(--space) * .5);
  --space-1: calc(var(--space) * .75);
  --space1: calc(var(--space) * 1.5);
  --space2: calc(var(--space) * 2);
  --space3: calc(var(--space) * 3);
}
""", layer='tokens', doc='A spacing scale derived from one application-wide base value.')

space [tokens]: A spacing scale derived from one application-wide base value.

In [ ]:
demo(swatches('padding', ['space-3', 'space-2', 'space-1', 'space', 'space1', 'space2', 'space3']))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }
:root, [danger], [warning], [success], [info], [project], mark {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 + 0.03348*cos((1*var(--hue) - 339.18)*1deg) + 0.01411*cos((2*var

## Radius

Three sizes derive from `--radius`, on every element as with space, so changing the base adjusts the application's general roundness, or one region's. `--radius-full` stays independent: a pill is a shape, not another step on the scale.

In [ ]:
reg.rule('radius', r"""
:root { --radius: .5rem; --radius-full: 9999px; }
*, ::before, ::after {
  --radius-1: calc(var(--radius) * .5);
  --radius1: calc(var(--radius) * 2);
}
""", layer='tokens', doc='Three corner radii derived from one base value, plus an independent pill radius.')

radius [tokens]: Three corner radii derived from one base value, plus an independent pill radius.

In [ ]:
demo(swatches('border-radius', ['radius-1', 'radius', 'radius1', 'radius-full']))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }
:root, [danger], [warning], [success], [info], [project], mark {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 + 0.03348*cos((1*var(--hue) - 339.18)*1deg) + 0.01411*cos((2*var

## Motion

`--motion` sets the shared duration, and `--ease` the easing curve. The spin that busy controls share is defined here too, so any component can turn it. Components use the duration directly or multiply it for longer effects. Temporarily increasing `--motion` lets us inspect transitions in slow motion. Scrolling the page, by a link to an anchor or a control taking focus, is smooth.

A reduced-motion preference sets the duration to zero for effects that consume it, and scrolling back to instant. Components must remain functional without motion and must not depend on an animation finishing to perform essential work.

In [ ]:
reg.rule('motion', r"""
:root {
  --motion: 150ms;
  --ease: cubic-bezier(.2, 0, 0, 1);
}
html { scroll-behavior: smooth; }
@media (prefers-reduced-motion: reduce) {
  :root { --motion: 0ms; }
  html { scroll-behavior: auto; }
}
@keyframes aui-spin { to { rotate: 1turn; } }
""", layer='tokens', doc='A shared duration and easing curve, and smooth scrolling, all stilled under reduced motion; and the spin shared by busy controls.')

motion [tokens]: A shared duration and easing curve, with zero duration under reduced motion, and the spin shared by busy controls.

## Elevation

Depth is relief under one light, from the top left. A raised surface has a light rim along its top and left edges and a dark rim along its bottom and right, plus a small lift beneath it. A well is the same lighting inverted, with a shadow inside its top edge. The rims are lighter and darker than the surface's own face, so we derive them from `--face` rather than from a rung: a rung can be lighter than the face in one scheme and darker in the other.

`--relief` controls the strength of this treatment. At `1` the rims and shadows have their full opacity, `.5` softens them, and `0` removes them. Components also use it to scale pressed displacement. It is declared on `:root` and inherited, so an application can adjust the whole page or one component without rewriting its state rules. Faces, borders, and state colours remain independent of relief. `--face` and `--relief` are the inputs and each element derives its own rims and shadows from them, so `--relief: 0` on a region flattens everything in it, while changing one component's treatment is a `box-shadow` on that component.

The rims must be derived on the element whose face they describe. We declare them on every element and on `::before` and `::after`, with `--face` defaulting to the raised rung. A raised surface combines `--bevel` and `--raised`; a well combines `--bevel-in` and `--sunken`. Translucent black keeps the shadows independent of colour contexts.

`--floating` separates an overlay from the page beneath it. It stays independent of relief, so a flat theme can retain depth beneath dialogs and menus.

In [ ]:
reg.rule('elevation', r"""
:root {
  --face: var(--n-2); --relief: 1;
  --_shadow-color: light-dark(rgb(0 0 0 / .12), rgb(0 0 0 / .5));
  --floating: 0 4px 12px var(--_shadow-color), 0 12px 32px var(--_shadow-color);
}
*, ::before, ::after {
  --hi: oklch(from var(--face) calc(l + .16) c h / var(--relief));
  --lo: oklch(from var(--face) calc(l - .16) c h / var(--relief));
  --_relief-shadow: light-dark(rgb(0 0 0 / calc(.12 * var(--relief))), rgb(0 0 0 / calc(.5 * var(--relief))));
  --bevel: inset 1px 1px 0 var(--hi), inset -1px -1px 0 var(--lo);
  --bevel-in: inset 1px 1px 0 var(--lo), inset -1px -1px 0 var(--hi);
  --raised: 0 2px 4px var(--_relief-shadow);
  --sunken: inset 0 2px 4px var(--_relief-shadow);
}
""", layer='tokens', doc='Face-derived rims and raised and sunken shadows scaled by relief, with independent floating shadows.')

elevation [tokens]: Face-derived rims and raised and sunken shadows scaled by relief, with independent floating shadows.

In [ ]:
demo(swatches('box-shadow', ['bevel', 'raised', 'bevel-in', 'sunken', 'floating']))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }
:root, [danger], [warning], [success], [info], [project], mark {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 + 0.03348*cos((1*var(--hue) - 339.18)*1deg) + 0.01411*cos((2*var

## Customization

Application overrides use the default `app` layer, leaving the library's entries intact. Set the base values on `:root` for application-wide changes, or on a region for local ones. Colour is the exception: a local hue is a context declared with `reg.context`, since the ladder is declared per context rather than derived on every element. Here we choose a green hue, larger text, tighter spacing, and squarer corners:

In [ ]:
reg.rule('app-theme', r"""
:root { --hue: 145; --text: 1.125rem; --space: .875rem; --radius: .25rem; }
""", doc='Application-wide colour, type size, spacing, and roundness.')

app-theme [app]: Application-wide colour, type size, spacing, and roundness.

The same swatches now show the theme:

In [ ]:
demo(swatches('background', rungs), swatches('font-size', ['text-1', 'text', 'text1', 'text2', 'text3']),
    swatches('padding', ['space-3', 'space-2', 'space-1', 'space', 'space1', 'space2', 'space3']),
    swatches('border-radius', ['radius-1', 'radius', 'radius1', 'radius-full']))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }
:root, [danger], [warning], [success], [info], [project], mark {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 + 0.03348*cos((1*var(--hue) - 339.18)*1deg) + 0.01411*cos((2*var

A region can have a scale of its own. Setting the base tokens on it rederives every step inside it. Both panels read `--text` for their font size, since text inherits its size rather than reading the token, and differ only in their inputs:


In [ ]:
def panel(title, style=''):
    "A heading, a line of text, and three spacing steps, styled from the tokens"
    return Div(Div(title, style='font-size:var(--text2);font-weight:var(--weight1)'), Div('Body text at the region\'s size.'),
        swatches('padding', ['space-2', 'space', 'space1']), style=f'font-size:var(--text);padding:var(--space);display:grid;gap:var(--space-2);{style}')

demo(Div(panel('Normal'), panel('Compact', style='--space:.75rem;--text:.875rem'), style='display:grid;grid-template-columns:1fr 1fr'))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }
:root, [danger], [warning], [success], [info], [project], mark {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 + 0.03348*cos((1*var(--hue) - 339.18)*1deg) + 0.01411*cos((2*var

A new colour context is one declaration. Put `project=True` on a FastHTML element to give it and its contents this purple palette:


In [ ]:
reg.context('[project]', 300)


In [ ]:
demo(swatches('background', rungs, project=True))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }
:root, [danger], [warning], [success], [info], [project], mark {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 + 0.03348*cos((1*var(--hue) - 339.18)*1deg) + 0.01411*cos((2*var

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()